# 머신러닝 기초 4장. 스태킹과 모형 비교

사회약학 연구방법 노트 '머신러닝 기초' 4장을 따라 하는 노트북입니다. 0–3장과 같은 가상 청구자료와 분할로 앞 장의 모형 일곱 개를 다시 만들고, 투표·평균·스태킹으로 합친 뒤, 같은 겹과 짝지은 비교, 반복 교차검증, 중첩 교차검증, 짝지은 부트스트랩으로 모형을 공정하게 비교하고 최종 모형을 골라 저장합니다. 셀을 위에서부터 차례로 실행하세요. 셀 2와 셀 9는 Colab에서 몇 분씩 걸립니다. 자료는 사이트가 만든 가상 자료이며 실제 환자 자료가 아닙니다.


## 준비

0장의 자료 불러오기와 분할, 0–3장의 전처리 함수 네 가지, 5겹 분할기(skf), 1장의 `boot_auc`를 한 셀에 모았습니다.

### 셀 1. 0–3장의 준비를 한 번에 (자료, 분할, 전처리 네 가지, skf, boot_auc)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, cross_val_predict)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (StandardScaler, OneHotEncoder,
                                   OrdinalEncoder)
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

BASE = "https://socialp-ajou.tecentriq12.workers.dev/data/"
UA = {"User-Agent": "Mozilla/5.0"}   # 사이트가 파이썬 기본 요청을 막아 브라우저처럼 보이게 함
df = pd.read_csv(BASE + "ml_claims.csv", storage_options=UA)
y = df["admit_2023"]
X = df.drop(columns=["id", "admit_2023", "cost_2023"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=2026)
cat_cols = ["insurance", "region", "smoking", "alcohol"]
num_cols = [c for c in X.columns if c not in cat_cols]

def cat_steps():
    """글자 열: 결측을 'missing' 범주로 채운 뒤 원-핫 (0-3장과 같음)"""
    return Pipeline([
        ("impute", SimpleImputer(strategy="constant",
                                 fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore",
                                 sparse_output=False))])

def make_pipe(model):
    """0장: 중앙값 대체 + 표준화 + 원-핫"""
    num_steps = Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())])
    prep = ColumnTransformer([("num", num_steps, num_cols),
                              ("cat", cat_steps(), cat_cols)])
    return Pipeline([("prep", prep), ("model", model)])

def make_tree_pipe(model):
    """1장: 중앙값 대체 + 원-핫 (표준화 없음)"""
    prep = ColumnTransformer(
        [("num", SimpleImputer(strategy="median"), num_cols),
         ("cat", cat_steps(), cat_cols)],
        verbose_feature_names_out=False)
    return Pipeline([("prep", prep), ("model", model)])

def make_nan_pipe(model):
    """3장: 숫자 열은 결측(NaN) 그대로, 글자 열만 원-핫"""
    prep = ColumnTransformer(
        [("num", "passthrough", num_cols),
         ("cat", cat_steps(), cat_cols)],
        verbose_feature_names_out=False).set_output(transform="pandas")
    return Pipeline([("prep", prep), ("model", model)])

def make_hgb_pipe(model):
    """3장 라 절: 글자 열은 정수 번호로(결측은 NaN) 앞에, 숫자 열은 그대로"""
    prep = ColumnTransformer(
        [("cat", OrdinalEncoder(handle_unknown="use_encoded_value",
                                unknown_value=np.nan), cat_cols)],
        remainder="passthrough", verbose_feature_names_out=False)
    return Pipeline([("prep", prep), ("model", model)])

def boot_auc(y_true, p, n_boot=1000, seed=2026):
    """1장 마 절: 시험 자료를 복원추출해 다시 구한 AUC n_boot개"""
    rng = np.random.default_rng(seed)
    yt, p, out = np.asarray(y_true), np.asarray(p), []
    for _ in range(n_boot):
        i = rng.integers(0, len(yt), len(yt))
        out.append(roc_auc_score(yt[i], p[i]))
    return np.array(out)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
print(X_train.shape, X_test.shape, y_train.sum(), y_test.sum())

## 가. 여러 모형을 합치는 방법

### 셀 2. 앞 장의 모형 일곱 개: 겹 밖 예측, 전체 학습, 시험 예측

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
                              HistGradientBoostingClassifier)
from xgboost import XGBClassifier

best_xgb = {"n_estimators": 200,            # 3장 마 절 Optuna가 고른 값
            "learning_rate": 0.08296006039772535,
            "max_depth": 1,
            "min_child_weight": 1.2474747561630017,
            "subsample": 0.6464876912292634,
            "colsample_bytree": 0.39626241159839404,
            "reg_lambda": 13.846707712074963,
            "reg_alpha": 0.0021251668855702768}
base = {
    "logistic": make_pipe(LogisticRegression(C=0.1, max_iter=1000)),
    "tree": make_tree_pipe(DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=300, random_state=2026)),
    "bagging": make_tree_pipe(BaggingClassifier(
        estimator=DecisionTreeClassifier(min_samples_leaf=20),
        n_estimators=200, random_state=2026, n_jobs=-1)),
    "forest": make_tree_pipe(RandomForestClassifier(
        n_estimators=500, max_features=20, min_samples_leaf=85,
        max_samples=0.7339919539656937, random_state=2026, n_jobs=-1)),
    "xgb_default": make_nan_pipe(XGBClassifier(random_state=2026)),
    "hgb": make_hgb_pipe(HistGradientBoostingClassifier(
        categorical_features=[0, 1, 2, 3], learning_rate=0.05,
        max_leaf_nodes=4, min_samples_leaf=50, max_iter=1000,
        early_stopping=True, n_iter_no_change=30, random_state=2026)),
    "xgb": make_nan_pipe(XGBClassifier(random_state=2026, **best_xgb))}

oof = pd.DataFrame(index=X_train.index)     # 겹 밖 예측 (학습 자료)
test_p = pd.DataFrame(index=X_test.index)   # 시험 자료 예측
secs = {}
for name, m in base.items():
    t0 = time.perf_counter()
    oof[name] = cross_val_predict(m, X_train, y_train, cv=skf,
                                  method="predict_proba")[:, 1]
    m.fit(X_train, y_train)                  # 학습 자료 전체로
    test_p[name] = m.predict_proba(X_test)[:, 1]
    secs[name] = time.perf_counter() - t0

folds = list(skf.split(X_train, y_train))   # 다섯 겹의 (학습, 검증) 번호

def fold_aucs(p):
    """겹 밖 예측 p로 겹마다 AUC (cross_val_score와 같은 값)"""
    p = np.asarray(p)
    return np.array([roc_auc_score(y_train.iloc[va], p[va])
                     for tr, va in folds])

fold_auc = pd.DataFrame({nm: fold_aucs(oof[nm]) for nm in base})
summary = pd.DataFrame({
    "CV AUC": fold_auc.mean(), "CV SD": fold_auc.std(ddof=0),
    "test AUC": [roc_auc_score(y_test, test_p[nm]) for nm in base],
    "seconds": pd.Series(secs).round().astype(int)})
summary.round(3)

### 셀 3. 환자 여섯 명으로 하드 투표, 소프트 투표, 가중 평균

In [ ]:
three = ["logistic", "forest", "xgb"]     # 서로 다른 집안의 세 모형
P = test_p[three]                         # 시험 자료 3,750명 × 3
votes = (P >= 0.5).sum(axis=1)            # 0.5 이상이라고 한 모형 수
toy = P.assign(votes=votes,
               hard=(votes >= 2).astype(int),          # 다수결
               soft=P.mean(axis=1),                    # 확률 평균
               weighted=(P * [1, 1, 2]).sum(axis=1) / 4,  # xgb 두 배
               admit=y_test)
print("hard vote = 1:", toy["hard"].sum(), "of", len(toy),
      "| admitted among them:", toy.loc[toy["hard"] == 1, "admit"].sum())
print("AUC  soft:", round(roc_auc_score(y_test, toy["soft"]), 3),
      "| number of votes:", round(roc_auc_score(y_test, votes), 3))
print("largest probability:", P.max().round(3).to_dict())
order = toy["soft"].sort_values(ascending=False).index
toy.loc[order[[0, 2, 5, 10, 150, 1875]]].round(3)

### 셀 4. VotingClassifier로 같은 일 하기

In [ ]:
from sklearn.ensemble import VotingClassifier

members = [(nm, base[nm]) for nm in three]   # (이름, 모형) 짝의 목록
vote_soft = VotingClassifier(members, voting="soft")
vote_hard = VotingClassifier(members, voting="hard")
vote_soft.fit(X_train, y_train)
vote_hard.fit(X_train, y_train)
p_soft = vote_soft.predict_proba(X_test)[:, 1]
print("soft = manual average :", np.allclose(p_soft, toy["soft"]))
print("hard = manual majority:",
      np.array_equal(vote_hard.predict(X_test), toy["hard"]))
try:
    vote_hard.predict_proba(X_test)
except AttributeError as e:
    print("AttributeError:", e)
cv_soft = fold_aucs(oof[three].mean(axis=1))  # 겹 밖 예측의 평균으로
print("soft vote: CV AUC", round(cv_soft.mean(), 4),
      "| test AUC", round(roc_auc_score(y_test, p_soft), 4))

### 셀 5. 기본 모형 예측의 상관과 두 모형을 평균한 이득

In [ ]:
import itertools

rho = oof.corr(method="spearman")      # 겹 밖 예측끼리의 순위 상관
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
im = ax[0].imshow(rho, vmin=0.4, vmax=1, cmap="viridis")
for i in range(len(rho)):
    for j in range(len(rho)):
        ax[0].text(j, i, f"{rho.iloc[i, j]:.2f}", ha="center",
                   va="center", fontsize=8,
                   color="black" if rho.iloc[i, j] > 0.75 else "white")
ax[0].set_xticks(range(len(rho)), rho.columns, rotation=45, ha="right")
ax[0].set_yticks(range(len(rho)), rho.columns)
ax[0].set_title("Spearman correlation, out-of-fold predictions")
fig.colorbar(im, ax=ax[0])

auc = summary["CV AUC"]
rows = []
for a, b in itertools.combinations(base, 2):
    both = fold_aucs(oof[[a, b]].mean(axis=1)).mean()  # 둘의 평균
    rows.append([a, b, rho.loc[a, b], abs(auc[a] - auc[b]),
                 both - max(auc[a], auc[b])])
pairs = pd.DataFrame(rows, columns=["model 1", "model 2", "rho",
                                    "AUC gap", "gain"])
close = pairs["AUC gap"] < 0.01             # 실력이 비슷한 짝
ax[1].scatter(pairs["rho"][~close], pairs["gain"][~close],
              color="lightgray", label="AUC gap >= 0.01")
ax[1].scatter(pairs["rho"][close], pairs["gain"][close],
              label="AUC gap < 0.01")
ax[1].axhline(0, color="gray", lw=0.8)
ax[1].set_xlabel("Spearman correlation of the two models")
ax[1].set_ylabel("CV AUC of their average - better one")
ax[1].legend()
plt.tight_layout()
pairs[close].sort_values("rho").round(4)

## 나. 스태킹 만들기

### 셀 6. 안쪽 예측으로 메타 모형을 배우면 (누설)

In [ ]:
from sklearn.ensemble import StackingClassifier

inside = pd.DataFrame({nm: m.predict_proba(X_train)[:, 1]  # 안쪽 예측
                       for nm, m in base.items()}, index=X_train.index)
print(pd.DataFrame({
    "inside": [roc_auc_score(y_train, inside[nm]) for nm in base],
    "out-of-fold": summary["CV AUC"]}, index=list(base)).T.round(3))

leaky = StackingClassifier(list(base.items()), cv="prefit",
                           final_estimator=LogisticRegression(max_iter=1000))
leaky.fit(X_train, y_train)        # 메타 모형이 안쪽 예측으로 배움
honest = LogisticRegression(max_iter=1000).fit(oof, y_train)  # 겹 밖
for name, meta, Z in [("inside", leaky, X_test),
                      ("out-of-fold", honest, test_p[list(base)])]:
    auc_t = roc_auc_score(y_test, meta.predict_proba(Z)[:, 1])
    print(f"test AUC, meta model trained on {name}: {auc_t:.3f}")
leaky3 = StackingClassifier([(nm, base[nm]) for nm in three], cv="prefit",
                            final_estimator=LogisticRegression(max_iter=1000))
leaky3.fit(X_train, y_train)       # 덜 외운 세 모형만 넣으면
honest3 = LogisticRegression(max_iter=1000).fit(oof[three], y_train)
print("three models only, inside vs out-of-fold:",
      round(roc_auc_score(y_test, leaky3.predict_proba(X_test)[:, 1]), 3),
      round(roc_auc_score(y_test, honest3.predict_proba(test_p[three])[:, 1]), 3))
coefs = pd.DataFrame({"inside": leaky.final_estimator_.coef_[0],
                      "out-of-fold": honest.coef_[0]}, index=list(base))
coefs.plot.barh(figsize=(6.5, 3.8))
plt.axvline(0, color="gray", lw=0.8)
plt.xlabel("Meta-model coefficient (probability input)")
plt.tight_layout()
coefs.round(2)

### 셀 7. StackingClassifier (기본 모형 셋, 메타 모형은 로지스틱 회귀)

In [ ]:
stack = StackingClassifier(
    estimators=[(nm, base[nm]) for nm in three],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=skf, stack_method="predict_proba")
t0 = time.perf_counter()
stack.fit(X_train, y_train)        # 3 모형 × (5겹 + 전체 1번) = 18번
t_stack = time.perf_counter() - t0
meta = stack.final_estimator_
print("coef:", meta.coef_.round(2), " intercept:",
      meta.intercept_.round(2), f" ({t_stack:.0f} s)")
same = LogisticRegression(max_iter=1000).fit(oof[three], y_train)
print("largest difference from fitting cell 2's oof:",
      np.abs(same.coef_ - meta.coef_).max().round(6))
p_stack = stack.predict_proba(X_test)[:, 1]
print("test AUC:", round(roc_auc_score(y_test, p_stack), 3))
params = stack.get_params()        # 바깥 이름__안쪽 이름
print(params["forest__model__min_samples_leaf"],
      params["final_estimator__C"], params["cv"])

### 셀 8. 메타 모형 조절: 입력의 척도와 C를 GridSearchCV로

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import FunctionTransformer

def to_logit(P):
    """확률을 로그 오즈로 (0과 1은 아주 조금 안쪽으로 당긴 뒤)"""
    P = np.clip(P, 1e-6, 1 - 1e-6)
    return np.log(P / (1 - P))

meta_pipe = Pipeline([("input", "passthrough"),
                      ("lr", LogisticRegression(max_iter=1000))])
grid = {"input": ["passthrough", FunctionTransformer(to_logit)],
        "lr__C": [0.01, 0.1, 1, 10, 100]}
gs_meta = GridSearchCV(meta_pipe, grid, cv=skf, refit="log loss",
                       scoring={"AUC": "roc_auc",
                                "log loss": "neg_log_loss"})
gs_meta.fit(oof[three], y_train)   # 메타 모형의 학습 자료 = 겹 밖 예측
best_lr = gs_meta.best_estimator_["lr"]
print("best C:", gs_meta.best_params_["lr__C"], "| coef:",
      best_lr.coef_.round(2), "| intercept:", best_lr.intercept_.round(2))

same_p = pd.DataFrame([[v] * 3 for v in [0.02, 0.05, 0.1, 0.2, 0.4]],
                      columns=three)    # 세 모형이 같은 확률을 낼 때
print(pd.DataFrame({
    "all three": same_p["xgb"],
    "probability input": meta.predict_proba(same_p.to_numpy())[:, 1],
    "log-odds input": gs_meta.predict_proba(same_p)[:, 1]}).round(3))

res = pd.DataFrame(gs_meta.cv_results_)
res["input"] = ["probability" if p == "passthrough" else "log odds"
                for p in res["param_input"]]
res["AUC"] = res["mean_test_AUC"]
res["log loss"] = -res["mean_test_log loss"]
res.pivot_table(index="param_lr__C", columns="input",
                values=["AUC", "log loss"]).round(4)

### 셀 9. 메타 모형을 바꾼 스태킹과 바깥 교차검증 (겹 속의 겹)

In [ ]:
from sklearn.base import clone

stack_lo = clone(stack).set_params(
    final_estimator=gs_meta.best_estimator_)   # 메타 모형만 바꿈
t0 = time.perf_counter()
cv_stack = cross_val_score(stack_lo, X_train, y_train, cv=skf,
                           scoring="roc_auc")  # 바깥 5겹 × 안쪽 5겹
t_outer = time.perf_counter() - t0
t0 = time.perf_counter()
stack_lo.fit(X_train, y_train)
t_fit = time.perf_counter() - t0
p_stack_lo = stack_lo.predict_proba(X_test)[:, 1]
print("fits: one stack", 3 * (5 + 1), "| outer CV", 5 * 3 * (5 + 1))
print("outer CV AUC:", cv_stack.round(4), round(cv_stack.mean(), 4),
      f"({t_outer:.0f} s)")
print("test AUC:", round(roc_auc_score(y_test, p_stack_lo), 3),
      f"(one fit {t_fit:.0f} s)")
w = pd.Series(stack_lo.final_estimator_["lr"].coef_[0], index=three)
print("meta coef (log-odds input):", w.round(2).to_dict())
print("GridSearchCV over the whole stack, 10 candidates:",
      f"about {10 * t_outer / 60:.0f} min")
w.plot.barh(figsize=(5, 2.2))
plt.axvline(0, color="gray", lw=0.8)
plt.xlabel("Meta-model coefficient (log-odds input)")
plt.tight_layout()

## 다. 모형을 공정하게 비교하기

### 셀 10. 겹 나누기만 바꿔 보기 (seed 10개)

In [ ]:
rows = []
for seed in range(10):
    cv_s = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    a = cross_val_score(base["logistic"], X_train, y_train, cv=cv_s,
                        scoring="roc_auc").mean()
    b = cross_val_score(base["xgb"], X_train, y_train, cv=cv_s,
                        scoring="roc_auc").mean()
    rows.append([seed, a, b, b - a])
seeds = pd.DataFrame(rows, columns=["fold seed", "logistic", "xgb",
                                    "xgb - logistic"])
print("unpaired (each model on its own split):",
      round(seeds["xgb"].min() - seeds["logistic"].max(), 4), "to",
      round(seeds["xgb"].max() - seeds["logistic"].min(), 4))
seeds.drop(columns="fold seed").describe().loc[
    ["mean", "std", "min", "max"]].round(4)

### 셀 11. 겹마다 AUC를 이어 보기 (짝지은 비교)

In [ ]:
fold_auc["soft vote"] = cv_soft
fold_auc["stack"] = cv_stack
show = ["logistic", "tree", "forest", "hgb", "soft vote", "stack", "xgb"]
fig, ax = plt.subplots(figsize=(7.5, 4))
for i in range(5):
    ax.plot(show, fold_auc.loc[i, show], marker="o",
            label=f"fold {i + 1}")
ax.set_ylabel("AUC in the validation fold")
ax.legend(fontsize=8, ncol=5)
plt.tight_layout()
diff = fold_auc[show].sub(fold_auc["xgb"], axis=0)  # 겹마다 xgb와의 차이
pd.DataFrame({"fold SD": fold_auc[show].std(ddof=0),
              "mean diff vs xgb": diff.mean(),
              "SD of diff": diff.std(ddof=0),
              "folds xgb higher": (diff < 0).sum()}).round(4)

### 셀 12. 반복 교차검증과 보정한 t 검정

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold
from scipy import stats

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5,
                               random_state=2026)   # 5겹 × 5번 = 25
rep = pd.DataFrame({nm: cross_val_score(base[nm], X_train, y_train,
                                        cv=rskf, scoring="roc_auc")
                    for nm in ["logistic", "tree", "hgb", "xgb"]})

def paired_t(d, corrected=True, ratio=1 / 4):
    """겹마다의 차이 d로 짝지은 t. corrected=True면 Nadeau-Bengio 보정
    (ratio = 검증 겹 크기 ÷ 학습 겹 크기, 5겹이면 1/4)"""
    J = len(d)
    var = d.var(ddof=1) * (1 / J + (ratio if corrected else 0))
    t = d.mean() / np.sqrt(var)
    return f"t = {t:.2f}, p = {2 * stats.t.sf(abs(t), J - 1):.2g}"

for a, b in [("xgb", "logistic"), ("xgb", "hgb")]:
    d = rep[a] - rep[b]
    print(f"{a} - {b}: mean {d.mean():.4f}, SD {d.std():.4f}")
    print("   usual    :", paired_t(d, corrected=False))
    print("   corrected:", paired_t(d))
rep.plot.box(figsize=(6, 3.6))
plt.ylabel("AUC, 5-fold CV repeated 5 times")
plt.tight_layout()
rep.describe().loc[["mean", "std", "min", "max"]].round(4)

### 셀 13. 중첩 교차검증 (XGBoost, 작은 격자 9개)

In [ ]:
from sklearn.model_selection import cross_validate

grid_x = {"model__max_depth": [1, 2, 3],
          "model__learning_rate": [0.03, 0.1, 0.3]}
xgb200 = make_nan_pipe(XGBClassifier(n_estimators=200,
                                     random_state=2026))
usual = GridSearchCV(xgb200, grid_x, cv=skf, scoring="roc_auc")
usual.fit(X_train, y_train)
print("usual :", usual.best_params_, round(usual.best_score_, 4))

inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
nested = cross_validate(
    GridSearchCV(xgb200, grid_x, cv=inner, scoring="roc_auc"),
    X_train, y_train, cv=skf, scoring="roc_auc", return_estimator=True)
for k, (s, g) in enumerate(zip(nested["test_score"],
                               nested["estimator"])):
    print(f"outer fold {k + 1}: {g.best_params_}")
    print(f"     inner best {g.best_score_:.4f}, outer {s:.4f}")
print(f"nested: {nested['test_score'].mean():.4f}")

### 셀 14. 같은 조합을 seed만 바꿔 20번 (1등을 고르는 일의 낙관)

In [ ]:
rows = []
for seed in range(20):
    m = make_nan_pipe(XGBClassifier(**{**best_xgb, "random_state": seed}))
    cv_auc = cross_val_score(m, X_train, y_train, cv=skf,
                             scoring="roc_auc").mean()
    m.fit(X_train, y_train)
    rows.append([seed, cv_auc,
                 roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])])
runs = pd.DataFrame(rows, columns=["seed", "CV AUC", "test AUC"])
top = runs.loc[runs["CV AUC"].idxmax()]
print("best of 20 by CV: seed", int(top["seed"]),
      top[["CV AUC", "test AUC"]].round(4).to_dict())
print("its test AUC ranks", int(runs["test AUC"].rank(ascending=False)
                                [top.name]), "of 20")
print("mean of 20      :",
      runs[["CV AUC", "test AUC"]].mean().round(4).to_dict())
print("seed 2026 (ch 3):", summary.loc["xgb", ["CV AUC", "test AUC"]]
      .astype(float).round(4).to_dict())

### 셀 15. 시험 자료에서 짝지은 부트스트랩과 DeLong 검정

In [ ]:
test_p["soft vote"] = p_soft
test_p["stack"] = p_stack_lo
boots = {nm: boot_auc(y_test, test_p[nm]) for nm in test_p}  # 같은 1,000벌

def delong(y, p1, p2):
    """DeLong(1988): 같은 사람들로 구한 두 AUC의 차이, 표준오차, p"""
    y = np.asarray(y)
    parts = []
    for p in (np.asarray(p1), np.asarray(p2)):
        pos, neg = p[y == 1], p[y == 0]
        win = (pos[:, None] > neg) + 0.5 * (pos[:, None] == neg)
        parts.append((win.mean(axis=1), win.mean(axis=0)))
    (v1, w1), (v2, w2) = parts     # v: 입원한 사람별, w: 안 한 사람별
    se = np.sqrt(np.var(v1 - v2, ddof=1) / len(v1)
                 + np.var(w1 - w2, ddof=1) / len(w1))
    d = v1.mean() - v2.mean()
    return d, se, 2 * stats.norm.sf(abs(d) / se)

rows = []
for a in ["stack", "soft vote", "hgb", "bagging", "forest", "logistic"]:
    d = boots[a] - boots["xgb"]          # 같은 표본에서의 차이
    diff, se, p = delong(y_test, test_p[a], test_p["xgb"])
    rows.append([a + " - xgb", diff, *np.percentile(d, [2.5, 97.5]),
                 d.std(ddof=1), se, p])
pd.DataFrame(rows, columns=["pair", "diff", "boot low", "boot high",
                            "boot SE", "DeLong SE", "DeLong p"]).round(3)

## 라. 최종 모형 고르기와 보고

### 셀 16. 모형 비교 표와 ROC 곡선

In [ ]:
from sklearn.metrics import roc_curve

order = ["logistic", "tree", "bagging", "forest", "xgb_default", "hgb",
         "xgb", "soft vote", "stack"]
secs["soft vote"] = sum(secs[nm] for nm in three)
secs["stack"] = t_outer + t_fit
rows = []
fig = plt.figure(figsize=(5.6, 5))
for nm in order:
    p = test_p[nm]
    lo, hi = np.percentile(boots[nm], [2.5, 97.5])
    rows.append([nm, fold_auc[nm].mean(), fold_auc[nm].std(ddof=0),
                 roc_auc_score(y_test, p), lo, hi, log_loss(y_test, p),
                 brier_score_loss(y_test, p), round(secs[nm])])
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, lw=1.1, label=nm)
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("1 - specificity")
plt.ylabel("Sensitivity")
plt.legend(fontsize=7)
table = pd.DataFrame(rows, columns=["model", "CV AUC", "CV SD",
                                    "test AUC", "CI low", "CI high",
                                    "log loss", "Brier", "seconds"])
table = table.set_index("model")
table.round(4)

### 셀 17. 한 표준오차 규칙

In [ ]:
best = table["CV AUC"].idxmax()
se = fold_auc[best].std(ddof=1) / np.sqrt(5)  # 겹 5개 평균의 표준오차
cut = table.loc[best, "CV AUC"] - se
print(f"best: {best}, CV AUC {table.loc[best, 'CV AUC']:.4f},",
      f"SE {se:.4f} -> keep models with CV AUC >= {cut:.4f}")
table.assign(within_1SE=table["CV AUC"] >= cut).sort_values(
    "CV AUC", ascending=False)[["CV AUC", "CV SD", "within_1SE"]].round(4)

### 셀 18. 교정 그림과 Brier 점수 (맛보기)

In [ ]:
from sklearn.calibration import calibration_curve

test_p["stack, probability meta"] = p_stack     # 셀 7의 스태킹
fig, ax = plt.subplots(1, 2, figsize=(10, 4.6))
for nm in ["logistic", "xgb", "stack", "stack, probability meta"]:
    obs, pred = calibration_curve(y_test, test_p[nm], n_bins=10,
                                  strategy="quantile")
    for a in ax:                                # 같은 점을 두 그림에
        a.plot(pred, obs, marker="o", ms=4, label=f"{nm} "
               f"(Brier {brier_score_loss(y_test, test_p[nm]):.4f})")
for a, lim in zip(ax, [0.3, 0.08]):             # 오른쪽은 아래쪽 확대
    a.plot([0, lim], [0, lim], "--", color="gray")
    a.set_xlim(0, lim)
    a.set_ylim(0, lim)
    a.set_xlabel("Mean predicted probability (10 groups)")
ax[0].set_ylabel("Observed admission rate")
ax[0].legend(fontsize=7)
ax[1].set_title("Zoom: probabilities below 0.08")
plt.tight_layout()
flat = np.full(len(y_test), y_train.mean())     # 모두에게 4.08%
print("Brier, 4.08% for everyone:",
      round(brier_score_loss(y_test, flat), 4))

### 셀 19. 최종 모형: 학습 자료 전체로 다시 맞추고 저장하기

In [ ]:
import joblib
import sklearn
import xgboost

final = clone(base["xgb"]).fit(X_train, y_train)  # 학습 자료 전체로 다시
p_final = final.predict_proba(X_test)[:, 1]       # 시험 자료는 한 번
lo, hi = np.percentile(boot_auc(y_test, p_final), [2.5, 97.5])
print(f"test AUC {roc_auc_score(y_test, p_final):.3f}"
      f" (95% CI {lo:.3f} to {hi:.3f})")
joblib.dump(final, "admit_model_final.joblib")
again = joblib.load("admit_model_final.joblib")
print("same after loading:",
      np.array_equal(again.predict_proba(X_test)[:, 1], p_final))
print("scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__)

### 셀 20. 부분 의존 그림: 개별 모형과 스태킹

In [ ]:
from sklearn.inspection import partial_dependence

X_pdp = X_train.sample(1000, random_state=2026)
X_pdp = X_pdp.astype({c: float for c in num_cols})  # 정수 열을 실수로
show_pd = {"logistic": base["logistic"], "forest": base["forest"],
           "xgb": base["xgb"], "stack": stack_lo}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
pdv = {}
for a, f in zip(ax, ["age", "n_ed"]):
    for nm, m in show_pd.items():
        r = partial_dependence(m, X_pdp, [f], percentiles=(0.01, 0.99),
                               grid_resolution=30)
        pdv[f, nm] = pd.Series(r["average"][0],
                               index=r["grid_values"][0])
        a.plot(r["grid_values"][0], r["average"][0], marker=".",
               label=nm)
    a.set_xlabel(f)
ax[0].set_ylabel("Average predicted probability")
ax[0].legend()
plt.tight_layout()
pd.DataFrame({nm: pdv["n_ed", nm] for nm in show_pd}).round(3)

## 과제 정답

**과제 1.** 셀 3의 세 모형에 가중치 [1, 1, 1], [1, 1, 2], [1, 1, 4], [0, 0, 1]을 주어 겹 밖 예측으로 교차검증 AUC를, 시험 예측으로 시험 AUC를 구합니다.

### 셀 21. 과제 1 정답. 가중치를 바꾼 소프트 투표

In [ ]:
for w in [[1, 1, 1], [1, 1, 2], [1, 1, 4], [0, 0, 1]]:
    p_cv = (oof[three] * w).sum(axis=1) / sum(w)    # 겹 밖 예측의 가중 평균
    p_te = (test_p[three] * w).sum(axis=1) / sum(w)
    print(w, "CV AUC", round(fold_aucs(p_cv).mean(), 4),
          "| test AUC", round(roc_auc_score(y_test, p_te), 4))

**과제 2.** 기본 모형 일곱 개의 겹 밖 예측으로 셀 8에서 고른 메타 모형을 맞추고 계수와 AUC를 봅니다.

### 셀 22. 과제 2 정답. 기본 모형 일곱 개로 스태킹

In [ ]:
seven = list(base)
meta7 = clone(gs_meta.best_estimator_).fit(oof[seven], y_train)
cv7 = cross_val_score(clone(gs_meta.best_estimator_), oof[seven],
                      y_train, cv=skf, scoring="roc_auc")
p7 = meta7.predict_proba(test_p[seven])[:, 1]
print(pd.Series(meta7["lr"].coef_[0], index=seven).round(2).to_dict())
print("meta-level CV AUC: 7 models", round(cv7.mean(), 4),
      "| 3 models", round(gs_meta.cv_results_["mean_test_AUC"]
                           [gs_meta.best_index_], 4))
print("test AUC: 7 models", round(roc_auc_score(y_test, p7), 4),
      "| 3 models", round(roc_auc_score(y_test, p_stack_lo), 4))

**과제 3.** 로지스틱 회귀, 히스토그램 부스팅, XGBoost로 스태킹을 만들어 바깥 교차검증과 시험 자료에서 XGBoost 하나와 짝지어 비교합니다.

### 셀 23. 과제 3 정답. 다른 세 모형으로 스태킹하고 공정하게 비교

In [ ]:
stack_b = StackingClassifier(
    [(nm, base[nm]) for nm in ["logistic", "hgb", "xgb"]],
    final_estimator=gs_meta.best_estimator_, cv=skf)
cv_b = cross_val_score(stack_b, X_train, y_train, cv=skf,
                       scoring="roc_auc")
d_cv = cv_b - fold_auc["xgb"].to_numpy()          # 같은 겹에서의 차이
print("outer CV AUC", round(cv_b.mean(), 4), "| diff vs xgb by fold",
      d_cv.round(4), "mean", round(d_cv.mean(), 4))
p_b = stack_b.fit(X_train, y_train).predict_proba(X_test)[:, 1]
d = boot_auc(y_test, p_b) - boots["xgb"]
diff, se, p = delong(y_test, p_b, test_p["xgb"])
print("test AUC", round(roc_auc_score(y_test, p_b), 4),
      "| diff", round(diff, 4), "boot 95% CI",
      np.percentile(d, [2.5, 97.5]).round(3), "| DeLong p", round(p, 3))
print(pd.Series(stack_b.final_estimator_["lr"].coef_[0],
                index=["logistic", "hgb", "xgb"]).round(2).to_dict())